In [0]:
data = [
    (101, "C001", "Laptop", "Electronics", "Hyderabad", "2024-01-01", "50000", 1),
    (102, "C002", "Mobile", "Electronics", "Chennai", "2024-01-02", None, 2),
    (103, "C001", "Tablet", "Electronics", "Hyderabad", "2024-01-03", "20000", 1),
    (104, "C003", "Laptop", "Electronics", "Delhi", "2024-01-04", "55000", 1),
    (105, "C002", "Mobile", "Electronics", "Chennai", "2024-01-05", "18000", 1),
    (106, "C004", "Watch", "Accessories", "Mumbai", "2024-01-06", "8000", 1),
    (103, "C001", "Tablet", "Electronics", "Hyderabad", "2024-01-07", "22000", 1),  # updated record
    (107, "C005", "Headphones", "Accessories", None, "2024-01-08", "3000", 1),
    (108, "C006", "Laptop", "Electronics", "Bangalore", "2024-01-09", "-45000", 1),  # invalid amount
    (109, "C007", "Mobile", "Electronics", "Chennai", "2024-01-10", "15000", 2),
    (109, "C007", "Mobile", "Electronics", "Chennai", "2024-01-10", "15000", 2)   # duplicate
]

columns = ["order_id", "customer_id", "product", "category", "city", "date", "amount", "quantity"]

In [0]:
from pyspark.sql import SparkSession

# Create DataFrame
data = [
    (101, "C001", "Laptop", "Electronics", "Hyderabad", "2024-01-01", "50000", 1),
    (102, "C002", "Mobile", "Electronics", "Chennai", "2024-01-02", None, 2),
    (103, "C001", "Tablet", "Electronics", "Hyderabad", "2024-01-03", "20000", 1),
    (104, "C003", "Laptop", "Electronics", "Delhi", "2024-01-04", "55000", 1),
    (105, "C002", "Mobile", "Electronics", "Chennai", "2024-01-05", "18000", 1),
    (106, "C004", "Watch", "Accessories", "Mumbai", "2024-01-06", "8000", 1),
    (103, "C001", "Tablet", "Electronics", "Hyderabad", "2024-01-07", "22000", 1),
    (107, "C005", "Headphones", "Accessories", None, "2024-01-08", "3000", 1),
    (108, "C006", "Laptop", "Electronics", "Bangalore", "2024-01-09", "-45000", 1),
    (109, "C007", "Mobile", "Electronics", "Chennai", "2024-01-10", "15000", 2),
    (109, "C007", "Mobile", "Electronics", "Chennai", "2024-01-10", "15000", 2)
]

columns = ["order_id","customer_id","product","category","city","date","amount","quantity"]

df = spark.createDataFrame(data, columns)

# Write to Bronze
df.write.format("delta") \
.mode("overwrite") \
.saveAsTable("bronze_orders")

In [0]:
from pyspark.sql.functions import col, when, to_date, row_number
from pyspark.sql.window import Window

df = spark.table("bronze_orders")
df = df.withColumn(
    "amount",
    when(col("amount").isNull(), 1000).otherwise(col("amount"))
).withColumn(
    "city",
    when(col("city").isNull(), "Unknown").otherwise(col("city"))
)
df = df.withColumn("amount", col("amount").cast("int")) \
       .withColumn("date", to_date(col("date")))

df = df.filter(col("amount") > 0)
window = Window.partitionBy("order_id").orderBy(col("date").desc())

df = df.withColumn("rn", row_number().over(window)) \
       .filter(col("rn") == 1) \
       .drop("rn")
df.write.format("delta") \
.mode("overwrite") \
.saveAsTable("silver_orders")

In [0]:
df = spark.table("bronze_orders").show()

+--------+-----------+----------+-----------+---------+----------+------+--------+
|order_id|customer_id|   product|   category|     city|      date|amount|quantity|
+--------+-----------+----------+-----------+---------+----------+------+--------+
|     101|       C001|    Laptop|Electronics|Hyderabad|2024-01-01| 50000|       1|
|     102|       C002|    Mobile|Electronics|  Chennai|2024-01-02|  NULL|       2|
|     103|       C001|    Tablet|Electronics|Hyderabad|2024-01-03| 20000|       1|
|     104|       C003|    Laptop|Electronics|    Delhi|2024-01-04| 55000|       1|
|     105|       C002|    Mobile|Electronics|  Chennai|2024-01-05| 18000|       1|
|     106|       C004|     Watch|Accessories|   Mumbai|2024-01-06|  8000|       1|
|     103|       C001|    Tablet|Electronics|Hyderabad|2024-01-07| 22000|       1|
|     107|       C005|Headphones|Accessories|     NULL|2024-01-08|  3000|       1|
|     108|       C006|    Laptop|Electronics|Bangalore|2024-01-09|-45000|       1|
|   

In [0]:
from pyspark.sql.functions import col, when

df = spark.table("bronze_orders")
df = df.withColumn(
    "amount",
    when(col("amount").isNull(), 1000).otherwise(col("amount"))
).withColumn(
    "city",
    when(col("city").isNull(), "Unknown").otherwise(col("city"))
)
df.show()

+--------+-----------+----------+-----------+---------+----------+------+--------+
|order_id|customer_id|   product|   category|     city|      date|amount|quantity|
+--------+-----------+----------+-----------+---------+----------+------+--------+
|     101|       C001|    Laptop|Electronics|Hyderabad|2024-01-01| 50000|       1|
|     102|       C002|    Mobile|Electronics|  Chennai|2024-01-02|  1000|       2|
|     103|       C001|    Tablet|Electronics|Hyderabad|2024-01-03| 20000|       1|
|     104|       C003|    Laptop|Electronics|    Delhi|2024-01-04| 55000|       1|
|     105|       C002|    Mobile|Electronics|  Chennai|2024-01-05| 18000|       1|
|     106|       C004|     Watch|Accessories|   Mumbai|2024-01-06|  8000|       1|
|     103|       C001|    Tablet|Electronics|Hyderabad|2024-01-07| 22000|       1|
|     107|       C005|Headphones|Accessories|  Unknown|2024-01-08|  3000|       1|
|     108|       C006|    Laptop|Electronics|Bangalore|2024-01-09|-45000|       1|
|   

In [0]:
from pyspark.sql.functions import to_date, col

df = spark.table("bronze_orders")
df = df.withColumn("amount", col("amount").cast("int")) \
       .withColumn("date", to_date(col("date")))
df.show()

+--------+-----------+----------+-----------+---------+----------+------+--------+
|order_id|customer_id|   product|   category|     city|      date|amount|quantity|
+--------+-----------+----------+-----------+---------+----------+------+--------+
|     101|       C001|    Laptop|Electronics|Hyderabad|2024-01-01| 50000|       1|
|     102|       C002|    Mobile|Electronics|  Chennai|2024-01-02|  NULL|       2|
|     103|       C001|    Tablet|Electronics|Hyderabad|2024-01-03| 20000|       1|
|     104|       C003|    Laptop|Electronics|    Delhi|2024-01-04| 55000|       1|
|     105|       C002|    Mobile|Electronics|  Chennai|2024-01-05| 18000|       1|
|     106|       C004|     Watch|Accessories|   Mumbai|2024-01-06|  8000|       1|
|     103|       C001|    Tablet|Electronics|Hyderabad|2024-01-07| 22000|       1|
|     107|       C005|Headphones|Accessories|     NULL|2024-01-08|  3000|       1|
|     108|       C006|    Laptop|Electronics|Bangalore|2024-01-09|-45000|       1|
|   

In [0]:
df = df.filter(col("amount") > 0)
df.show()

+--------+-----------+----------+-----------+---------+----------+------+--------+
|order_id|customer_id|   product|   category|     city|      date|amount|quantity|
+--------+-----------+----------+-----------+---------+----------+------+--------+
|     101|       C001|    Laptop|Electronics|Hyderabad|2024-01-01| 50000|       1|
|     103|       C001|    Tablet|Electronics|Hyderabad|2024-01-03| 20000|       1|
|     104|       C003|    Laptop|Electronics|    Delhi|2024-01-04| 55000|       1|
|     105|       C002|    Mobile|Electronics|  Chennai|2024-01-05| 18000|       1|
|     106|       C004|     Watch|Accessories|   Mumbai|2024-01-06|  8000|       1|
|     103|       C001|    Tablet|Electronics|Hyderabad|2024-01-07| 22000|       1|
|     107|       C005|Headphones|Accessories|     NULL|2024-01-08|  3000|       1|
|     109|       C007|    Mobile|Electronics|  Chennai|2024-01-10| 15000|       2|
|     109|       C007|    Mobile|Electronics|  Chennai|2024-01-10| 15000|       2|
+---

In [0]:
from pyspark.sql.window import Window
from pyspark.sql.functions import row_number

window = Window.partitionBy("order_id").orderBy(col("date").desc())

df = df.withColumn("rn", row_number().over(window)) \
       .filter(col("rn") == 1) \
       .drop("rn")
df.show()

+--------+-----------+----------+-----------+---------+----------+------+--------+
|order_id|customer_id|   product|   category|     city|      date|amount|quantity|
+--------+-----------+----------+-----------+---------+----------+------+--------+
|     101|       C001|    Laptop|Electronics|Hyderabad|2024-01-01| 50000|       1|
|     103|       C001|    Tablet|Electronics|Hyderabad|2024-01-07| 22000|       1|
|     104|       C003|    Laptop|Electronics|    Delhi|2024-01-04| 55000|       1|
|     105|       C002|    Mobile|Electronics|  Chennai|2024-01-05| 18000|       1|
|     106|       C004|     Watch|Accessories|   Mumbai|2024-01-06|  8000|       1|
|     107|       C005|Headphones|Accessories|     NULL|2024-01-08|  3000|       1|
|     109|       C007|    Mobile|Electronics|  Chennai|2024-01-10| 15000|       2|
+--------+-----------+----------+-----------+---------+----------+------+--------+



In [0]:
df.write.format("delta") \
.mode("overwrite") \
.saveAsTable("silver_orders")

3. GOLD LAYER (Business Insights)

In [0]:
from pyspark.sql.functions import sum

sales_product = spark.table("silver_orders") \
    .groupBy("product") \
    .agg(sum("amount").alias("total_sales"))

sales_product.show()

+----------+-----------+
|   product|total_sales|
+----------+-----------+
|    Laptop|     105000|
|    Tablet|      22000|
|    Mobile|      33000|
|     Watch|       8000|
|Headphones|       3000|
+----------+-----------+



In [0]:
sales_category = spark.table("silver_orders") \
    .groupBy("category") \
    .agg(sum("amount").alias("total_sales"))

sales_category.show()

+-----------+-----------+
|   category|total_sales|
+-----------+-----------+
|Electronics|     160000|
|Accessories|      11000|
+-----------+-----------+



In [0]:
sales_city= spark.table("silver_orders").groupBy("city").agg(sum("amount").alias("total_sales"))
sales_city.show()

+---------+-----------+
|     city|total_sales|
+---------+-----------+
|Hyderabad|      72000|
|    Delhi|      55000|
|  Chennai|      33000|
|   Mumbai|       8000|
|     NULL|       3000|
+---------+-----------+



Requirement 2: 

Customer Insights

Number of orders per customer

Top customers based on spending


In [0]:
orders_customer = spark.table("silver_orders") \
    .groupBy("customer_id") \
    .count()
orders_customer.show()

+-----------+-----+
|customer_id|count|
+-----------+-----+
|       C001|    2|
|       C003|    1|
|       C002|    1|
|       C004|    1|
|       C005|    1|
|       C007|    1|
+-----------+-----+



In [0]:
from pyspark.sql.functions import sum

customer_spending = spark.table("silver_orders") \
    .groupBy("customer_id") \
    .agg(sum("amount").alias("total_spent"))

customer_spending.show()

+-----------+-----------+
|customer_id|total_spent|
+-----------+-----------+
|       C001|      72000|
|       C003|      55000|
|       C002|      18000|
|       C004|       8000|
|       C005|       3000|
|       C007|      15000|
+-----------+-----------+



In [0]:
from pyspark.sql.functions import to_date, col, row_number
from pyspark.sql.window import Window

data = [(101,"2024-01-01"), (103,"2024-01-03"), (103,"2024-01-07")]
df = spark.createDataFrame(data, ["order_id","date"]).withColumn("date", to_date(col("date")))

window = Window.partitionBy("order_id").orderBy(col("date").desc())
df.withColumn("rn", row_number().over(window)).filter(col("rn")==1).drop("rn").show()

+--------+----------+
|order_id|      date|
+--------+----------+
|     101|2024-01-01|
|     103|2024-01-07|
+--------+----------+



In [0]:
top_customer = customer_spending.orderBy(col("total_spent").desc()).limit(1)

top_customer.show()

+-----------+-----------+
|customer_id|total_spent|
+-----------+-----------+
|       C001|      72000|
+-----------+-----------+



In [0]:
top_product = sales_product.orderBy(col("total_sales").desc()).limit(1)

top_product.show()

+-------+-----------+
|product|total_sales|
+-------+-----------+
| Laptop|     105000|
+-------+-----------+



In [0]:
sales_product.write.format("delta").mode("overwrite").saveAsTable("gold_sales_product")
sales_category.write.format("delta").mode("overwrite").saveAsTable("gold_sales_category")
sales_city.write.format("delta").mode("overwrite").saveAsTable("gold_sales_city")
customer_spending.write.format("delta").mode("overwrite").saveAsTable("gold_customer_spending")